# Following the typed notes

First import and construct our network; we first deal with scalar values. 

In [20]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from flax import linen as nn
import numpy as np

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()

In [2]:
n = 1 # 1 dimensional input and output 
L = 18 # Number of layers; trying to emulate the paper

In [3]:
class MLP(nn.Module):
    """
    We first deal with something that's not exactly MLP, but close enough
    """
    num_units: int
    
    def setup(self):
        self.dense1 = nn.Dense(self.num_units)
        # self.dense2 = nn.Dense(self.num_units)
    
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.leaky_relu(f)
        # f = nn.tanh(f)
        # x = self.dense2(x)
        return x + f
    

class SimpleMLP(nn.Module):
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = [MLP(self.num_units) for _ in range(self.num_layers)]
        # self.classification_layer = nn.Dense(self.num_classes)

    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}', x)
            
        # Final layer to produce output
        # x = self.classification_layer(x)
        return x


In [4]:
model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
# Initialize the model parameters
rng = jax.random.PRNGKey(0)
input_shape = (5, n)  # Batch size of 1 and input dimension of 32
dummy_input = jnp.ones(input_shape)
params = model.init(rng, dummy_input)

# Forward pass
output = model.apply(params, dummy_input)
print("Output shape:", output.shape)  # Should be (1, num_classes)

Output shape: (5, 1)


In [15]:
# Print intermediate outputs using sow
n_samples = 1
x = jnp.ones((n_samples, n))

_, inter = model.apply(params, x, mutable=['intermediates'])
# inter

In [6]:
# params

## Compute a Jacobian 

Let's just use a single sample and compute this Jacobian 

See jacobian_comparison.py; this section will be deleted... 

In [7]:
def mse_loss(predictions, targets):
    return jnp.mean((predictions - targets) ** 2)

In [8]:
n_samples = 1
x = jnp.ones((n_samples, n)) 
y = jnp.ones((n_samples, n)) * .1
prediction = model.apply(params, x)
print(prediction)

[[494.08533]]


In [9]:
def loss_fn(params): 
    prediction = model.apply(params, x)
    loss = mse_loss(prediction, y)
    return loss
    
grad_fn = jax.grad(loss_fn)
    

In [10]:
# Compute the Jacobian matrix of the model's output with respect to the input
jacobian = jax.jacobian(model.apply, argnums=0)(params, x)
# jacobian

In [11]:
treemap_grad = jax.tree.map(lambda x: jnp.tensordot((prediction - y) * 2 / (n * n_samples), x), jacobian)
# treemap_grad

In [12]:
_, vjp_fn = jax.vjp(model.apply, params, x)
residuals = (prediction - y)
manual_grad = vjp_fn(residuals * 2 / (n * n_samples))[0] 
# manual_grad

In [13]:
real_grad = grad_fn(params)
# real_grad

In [14]:
# Can use this to verify that gradients are the same
(jax.tree.map(lambda x, y: x - y, treemap_grad, real_grad)) # TODO: seems wrong for higher L? 

{'params': {'layers_0': {'dense1': {'bias': <jax.Array([0.00097656], dtype=float32)>,
    'kernel': <jax.Array([[0.00097656]], dtype=float32)>}},
  'layers_1': {'dense1': {'bias': <jax.Array([0.00097656], dtype=float32)>,
    'kernel': <jax.Array([[0.00097656]], dtype=float32)>}},
  'layers_10': {'dense1': {'bias': <jax.Array([0.00390625], dtype=float32)>,
    'kernel': <jax.Array([[0.0625]], dtype=float32)>}},
  'layers_11': {'dense1': {'bias': <jax.Array([3.0517578e-05], dtype=float32)>,
    'kernel': <jax.Array([[0.00048828]], dtype=float32)>}},
  'layers_12': {'dense1': {'bias': <jax.Array([0.00097656], dtype=float32)>,
    'kernel': <jax.Array([[0.015625]], dtype=float32)>}},
  'layers_13': {'dense1': {'bias': <jax.Array([0.], dtype=float32)>,
    'kernel': <jax.Array([[0.00048828]], dtype=float32)>}},
  'layers_14': {'dense1': {'bias': <jax.Array([0.00048828], dtype=float32)>,
    'kernel': <jax.Array([[0.03125]], dtype=float32)>}},
  'layers_15': {'dense1': {'bias': <jax.Array([0.], dtype=float32)>,
    'kernel': <jax.Array([[0.]], dtype=float32)>}},
  'layers_16': {'dense1': {'bias': <jax.Array([-1.9073486e-06], dtype=float32)>,
    'kernel': <jax.Array([[0.]], dtype=float32)>}},
  'layers_17': {'dense1': {'bias': <jax.Array([0.], dtype=float32)>,
    'kernel': <jax.Array([[0.]], dtype=float32)>}},
  'layers_2': {'dense1': {'bias': <jax.Array([0.03125], dtype=float32)>,
    'kernel': <jax.Array([[0.03125]], dtype=float32)>}},
  'layers_3': {'dense1': {'bias': <jax.Array([0.00048828], dtype=float32)>,
    'kernel': <jax.Array([[0.00097656]], dtype=float32)>}},
  'layers_4': {'dense1': {'bias': <jax.Array([0.03125], dtype=float32)>,
    'kernel': <jax.Array([[0.046875]], dtype=float32)>}},
  'layers_5': {'dense1': {'bias': <jax.Array([0.015625], dtype=float32)>,
    'kernel': <jax.Array([[0.03125]], dtype=float32)>}},
  'layers_6': {'dense1': {'bias': <jax.Array([0.00018311], dtype=float32)>,
    'kernel': <jax.Array([[0.00097656]], dtype=float32)>}},
  'layers_7': {'dense1': {'bias': <jax.Array([0.00018311], dtype=float32)>,
    'kernel': <jax.Array([[0.00146484]], dtype=float32)>}},
  'layers_8': {'dense1': {'bias': <jax.Array([6.1035156e-05], dtype=float32)>,
    'kernel': <jax.Array([[0.]], dtype=float32)>}},
  'layers_9': {'dense1': {'bias': <jax.Array([0.00390625], dtype=float32)>,
    'kernel': <jax.Array([[0.03125]], dtype=float32)>}}}}

## Okay now we have three ways of calculating derivatives and such

Let's move on to the actual calculations...

We need a way of calculating $M_i$ and $K_i$ on a layer by layer basis

In [22]:
# Fake data
n_samples = 1
x = jnp.ones((n_samples, n))
# y = jnp.ones((n_samples, n)) * .1

# First, pretend to call the model
model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init( jax.random.PRNGKey(0), x)

predictions, intermediates = model.apply(params, x, mutable=['intermediates'])
# predictions, intermediates

In [23]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)
# layer.apply({'params': params['params']['layers_0']}, x)

# First define function; one input
def apply_layer(params, x): 
    assert len(x) == 1
    return jnp.squeeze(layer.apply(params, x))

# The issue with jax.grad is it doesn't automatically vectorize over batch dimension, so we do it manually with vmap and squeeze
K_i_one = jax.grad(apply_layer, 0)
K_i = jax.vmap(K_i_one, (None, 0)) # We don't allow vmapping over the params. I guess we could in theory... but it's easier to wrap my head around htis

M_i_one = jax.grad(apply_layer, 1)
M_i = jax.vmap(M_i_one, (None, 0))

In [24]:
# K_i({'params': params['params']['layers_4']}, intermediates['intermediates']['layer_5'][0])

In [25]:
# M_i({'params': params['params']['layers_0']}, x)

In [34]:
# dg du 
# Using np here to allow for inline edits... 
dgdu = np.eye(L + 1) # It's tridiagonal with ones down diagonal; will have to change once scaled up
for l in range(L):
    dgdu[l, l+1] =  -jnp.squeeze(M_i({'params': params['params'][f'layers_{l}']}, intermediates['intermediates'][f'layer_{l}'][0]))
treescope.display(dgdu)

In [27]:
# np.linalg.eigvals(dgdu)

In [28]:
# With these, let's build our little matrices 
p = sum(x.size for x in jax.tree.leaves(params))
dgdt = np.zeros((p, L + 1))

# This is harder to construct; need a mapping from dof to matrix... think about this for bigger problems
# Better way would be a matrix free operation
for i in range(L): 
    flattened, _ = jax.tree.flatten(
            K_i({'params': params['params'][f'layers_{i}']}, intermediates['intermediates'][f'layer_{i}'][0])
    )
    vals = np.array([jnp.squeeze(x) for x in flattened])

    # Can make this dynamic 
    dgdt[(2*i):(2*i + 2), i + 1] = vals
    
dgdt

array([[0.00000000e+00, 9.99999978e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 9.99999978e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 9.99999978e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 9.91382543e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 9.72128928e-01,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        9.99999978e-03, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        1.76084563e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.74564624e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 4.24246883e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.000

In [29]:
# If we do the total derivative, the last thing should be the Jacobian 
derivative = np.linalg.inv(dgdu.T) @ dgdt.T
derivative

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 9.99999978e-03,  9.99999978e-03,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 9.80578968e-03,  9.80578968e-03,  9.99999978e-03,
         9.91382543e-03, -6.12932735e-17, -5.95849642e-17,
        -2.47307808e-18, -4.35470883e-18, -5.08798054e-17,
        -8.88181410e-17, -7.14949196e-17, -3.03314968e-16,
        -1.45344230e-19, -1.09705079e-18, -1.47182362e-19,
        -1.09292926e-18, -1.49248826e-19, -1.09443321e-18,
         0.00000000e+00,  0.00000000e+00, -1.66528868e-17,
        -1.87265029e-16, -8.38483056e-20, -1.08684814e-18,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -6.80483118e-19, -4.24418897e-17,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.77615142e-02,  1.77615142e-02,  1.81132926e-02,
         1.79572025e-02,  1.00000000e+00,  9.72128928e-01,
        -4.47955880e-18, -7.88781172e-18, -9.21600826e-17,
        -1.60878902e-16, -1.29500843e-16, -5.49403291e-16,
        -2.63266263e-19, -1.98712024e-18, -2.66595726e-19,
        -1.97965480e-18, -2.70338772e-19, -1.98237895e-18,
         0.00000000e+00,  0.00000000e+00, -3.01638618e-17,
        -3.39198635e-16, -1.51876893e-19, -1.96863989e-18,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -1.23257901e-18, -7.68762386e-17,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.76081998e-02,  1.76081998e-02,  1.79569417e-02,
         1.78021990e-02,  9.91368168e-01,  9.63737674e-01,
         9.99999978e-03,  1.76084563e-02, -9.13645723e-17,
        -1.59490222e-16, -1.28383014e-16, -5.44660934e-16,
        -2.60993793e-19, -1.96996775e-18, -2.64294516e-19,
        -1.96256675e-18, -2.68005253e-19, -1.96526739e-18,
         0.00000000e+00,  0.00000000e+00, -2.99034924e-17,
        -3.36270729e-16, -1.50565917e-19, -1.95164692e-18,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -1.22193960e-18, -7.62126558e-17,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 4.27934556e-02,  4.27934556e-02,  4.36410081e-02,
         4.32649345e-02,  2.40933600e+00,  2.34218522e+00,
         2.43031401e-02,  4.27940790e-02,  1.00000000e+00,
         1.74564624e+00, -3.12011043e-16, -1.32369713e-15,
        -6.34296885e-19, -4.78764033e-18, -6.42318680e-19,
        -4.76965357e-18, -6.51336935e-19, -4.77621697e-18,
         0.00000000e+00,  0.00000000e+00, -7.26748783e-17,
        -8.17243483e-16, -3.65922466e-19, 

In [30]:
# Let's compare this with the Jacobian 
jacobian = jax.jacobian(model.apply, argnums=0)(params, x)
jacobian = jnp.array([jnp.squeeze(x) for x in jax.tree.flatten(jacobian)[0]])
jacobian 

<jax.Array float32(36,) ≈8.2e+01 ±1.2e+02 [≥0.032, ≤4.3e+02] nonzero:36
  <Arrayviz rendering>
| Device: GPU 0>

In [31]:
print(jnp.linalg.norm(jacobian - derivative[-1, :]))
jacobian - derivative[-1, :]

777.1270518601822


<jax.Array float64(36,) ≈8.5e-07 ±1.3e+02 [≥-2.7e+02, ≤2.7e+02] nonzero:36
  <Arrayviz rendering>
| Device: GPU 0>

In [32]:
derivative[-1, :]

array([4.98380056e+00, 4.98380056e+00, 5.08250800e+00, 5.03870982e+00,
       2.80595477e+02, 2.72774981e+02, 2.83038613e+00, 4.98387316e+00,
       1.16461746e+02, 2.03301009e+02, 6.54595520e+01, 2.77710109e+02,
       6.65373723e-01, 5.02220671e+00, 6.73788540e-01, 5.00333870e+00,
       6.83248636e-01, 5.01022367e+00, 4.39374455e+01, 3.17729810e+02,
       3.81177612e+01, 4.28641817e+02, 3.83850527e-01, 4.97549984e+00,
       1.47153049e+01, 1.89412692e+02, 1.49420640e-01, 5.01699066e+00,
       7.92181326e+00, 2.61948525e+02, 3.11519478e+00, 1.94295420e+02,
       3.15426891e-02, 5.00282692e+00, 1.00000000e+00, 1.56640198e+02])

# With a simple example done, we can do one of two things 

1. Play with the approximations for larger and larger L
2. Implement some sort of ASM
3. Expand so that we're not hardcoding

## Simple diagonal preconditioning 

Using a diagonal preconditioning doesn't work on dgdu; there is nto talking amongst the layers, and only the last layer has the correct jacobian. 

Though, one can interpret this as just applying Gauss-Newton on the last layer? e.g. if we do interpretation of basis, it does a good job? But I guess the last layer typically is linear anyways, so it's no use

In [305]:
np.linalg.inv(np.diag(np.diag(dgdu.T))) @ dgdt.T

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.01      , 0.01      , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.01      , 0.00991383, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 1.        ,
        0.97212893, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.01      , 0.01760846, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 1.        , 1.74564636,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        1.        , 4.24246883, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.01      , 0.07547949, 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.01      ,
        0.07425682, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.01      , 0.07332943]])

## Matrix of interest

It's actually dg/du dg/du.T (or is it dg/du.T dg/du?, but spectrally equivalent)

In [306]:
np.linalg.inv(dgdu @ dgdu.T)

array([[  1.        ,   0.9913826 ,   0.97212898,   1.76084576,
          1.74564645,   4.24246911,   7.547949  ,   7.42568163,
          7.33294391,   7.23141386],
       [  0.9913826 ,   1.98283946,   1.94433077,   3.52182342,
          3.49142365,   8.485256  ,  15.09646339,  14.85191949,
         14.6664371 ,  14.4633694 ],
       [  0.97212898,   1.94433077,   2.90656996,   5.26475551,
          5.21931106,  12.68456508,  22.56762463,  22.20205722,
         21.92478056,  21.62121571],
       [  1.76084576,   3.52182342,   5.26475551,  10.53620626,
         10.44525957,  25.38526123,  45.16394869,  44.43234898,
         43.87744304,  43.2699273 ],
       [  1.74564645,   3.49142365,   5.21931106,  10.44525957,
         11.35509791,  27.59645415,  49.09797177,  48.30264579,
         47.69940455,  47.03897092],
       [  4.24246911,   8.485256  ,  12.68456508,  25.38526123,
         27.59645415,  68.06805064, 121.10263189, 119.14092011,
        117.65299509, 116.0240021 ],
       [  7.547949  ,  15.09646339,  22.56762463,  45.16394869,
         49.09797177, 121.10263189, 216.45860815, 212.95224834,
        210.29273405, 207.38107516],
       [  7.42568163,  14.85191949,  22.20205722,  44.43234898,
         48.30264579, 119.14092011, 212.95224834, 210.50268719,
        207.87376493, 204.99559847],
       [  7.33294391,  14.6664371 ,  21.92478056,  43.87744304,
         47.69940455, 117.65299509, 210.29273405, 207.87376493,
        206.27767472, 203.4216073 ],
       [  7.23141386,  14.4633694 ,  21.62121571,  43.2699273 ,
         47.03897092, 116.0240021 , 207.38107516, 204.99559847,
        203.4216073 , 201.60508426]])

In [307]:
np.linalg.eigvalsh(np.linalg.inv(dgdu @ dgdu.T))

array([1.16544795e-01, 1.81946916e-01, 2.41601146e-01, 3.16529135e-01,
       3.50647297e-01, 5.62778484e-01, 8.52423746e-01, 1.69080348e+00,
       2.34641888e+00, 9.24033125e+02])

## Additive Scharz

Now let's do some additive Schwarz

In [308]:
def partition_with_overlap(L, N):
    """
    ChatGPT'd because I'm bad at LeetCode
    Partition the range [0, ..., L] into N overlapping subranges.
    
    Parameters:
        L (int): The maximum integer in the range.
        N (int): The number of partitions.
    
    Returns:
        List[int]: The partition points [c_0, c_1, ..., c_N].
    """
    # Compute the approximate size of each partition
    step = L / N
    
    # Generate the partition points
    c = [round(i * step) for i in range(N + 1)]
    
    return c
ND = 3 # For now; TODO: need a way to do this automatically; shouldn't be too hard but ugh

# Example usage:
# L = 9
# N = 3
print(partition_with_overlap(L, ND)) 

[0, 3, 6, 9]


In [309]:
# If we were to do a ras type preconditioner.... need restriction operators 

ND = 3 # For now; TODO: need a way to do this automatically; shouldn't be too hard but ugh

def make_matrices(L, ND): 
    """
    Given L layers and ND blocks, give the restriction, pou and q matrices for testing
    """
    restrictions = []
    pous = []
    qs = []

    cs = partition_with_overlap(L, ND)

    for d in range(ND): 
        time_steps = cs[d + 1] - cs[d] + 1
        r = np.zeros((time_steps, L + 1))
        for t in range(time_steps):
            r[t, cs[d] + t] = 1

        pou = np.eye(time_steps)
        if d == 0:
            pou[-1, -1] = 0.5
        elif d == ND - 1:
            pou[0, 0] = 0.5
        else: 
            pou[0, 0] = 0.5
            pou[-1, -1] = 0.5

        if d == 0:
            q = np.zeros((time_steps, L + 1))
            for i in range(time_steps): 
                q[i, i] = 1
        else:
            q = np.zeros((time_steps + 1, L + 1))
            for t in range(time_steps + 1): 
                q[t, cs[d] - 1 + t] = 1
        restrictions.append(r)
        pous.append(pou)
        qs.append(q)

    # assert np.linalg.norm(R1.T @ D1 @ R1 + R2.T @ D2 @ R2 + R3.T @ D3 @ R3 - np.eye(10)) < 1e-10

    return restrictions, pous, qs

restrictions, pous, qs = make_matrices(L, ND)
 

In [310]:
   
R1 = np.zeros((4, L + 1))
R2 = np.zeros((4, L + 1))
R3 = np.zeros((4, L + 1))
for i in range(4): 
    R1[i, i] = 1
    R2[i, 3 + i] = 1
    R3[i, 6 + i] = 1
treescope.display(
    (R1, R2, R3)
)

# Try 0.5 for now
D1 = np.eye(4)
D1[-1, -1] = 0.5
D2 = np.eye(4)
D2[-1, -1] = 0.5
D2[0, 0] = 0.5
D3 = np.eye(4)
D3[0, 0] = 0.5

Q1 = np.zeros((4, L + 1))
Q2 = np.zeros((5, L + 1))
Q3 = np.zeros((5, L + 1))
for i in range(4): 
    Q1[i, i] = 1
for i in range(5): 
    Q2[i, 2 + i] = 1
    Q3[i, 5 + i] = 1
treescope.display(
    (Q1, Q2, Q3)
)


# Make sure partition of unit matrix make sense (1.25 of Dolean book)
# assert np.linalg.norm(R1.T @ D1 @ R1 + R2.T @ D2 @ R2 + R3.T @ D3 @ R3 - np.eye(10)) < 1e-10

restrictions = [R1, R2, R3]
pou = [D1, D2, D3]
Qs = [Q1, Q2, Q3]

In [311]:
# Construct RAS preconditioner 
ras = np.zeros((L + 1, L + 1))
for i in range(ND): 
    R = restrictions[i]
    D = pous[i]
    ras += R.T @ D @ jnp.linalg.inv(R @ dgdu.T @ dgdu @ R.T) @ R
treescope.display(ras)

# Technically, this should be SPD, but then need to do the whole P^{1/2} business. 
np.linalg.eigvals(ras @  dgdu.T @ dgdu)

array([0.00808399+0.000000e+00j, 1.9900615 +0.000000e+00j,
       0.8380733 +0.000000e+00j, 1.1587238 +0.000000e+00j,
       0.99723864+0.000000e+00j, 0.99857175+0.000000e+00j,
       0.9985435 +8.651105e-05j, 0.9985435 -8.651105e-05j,
       0.9991681 +0.000000e+00j, 0.9992483 +0.000000e+00j],
      dtype=complex64)

In [312]:
# Construct RAS preconditioner 
rasq = np.zeros((L + 1, L + 1))
for i in range(ND): 
    R = restrictions[i]
    D = pou[i]
    Q = Qs[i]
    Js = np.linalg.inv(Q @ dgdu @ Q.T)
    rasq += R.T @ D @ (R @ Q.T @ Js @ Js.T @ Q @ R.T) @ R
treescope.display(rasq)

# Technically, this should be SPD, but then need to do the whole P^{1/2} business. 
jnp.linalg.eigvals(rasq @  dgdu.T @ dgdu)

# jax.Array complex64(10,)
  Array([1.0000000e+00+0.j, 1.0000000e+00+0.j, 4.4991646e+01+0.j,
         6.8269014e+00+0.j, 1.9443465e-02+0.j, 1.1409369e+00+0.j,
         1.0000178e+00+0.j, 1.0000002e+00+0.j, 1.0000012e+00+0.j,
         1.0000000e+00+0.j], dtype=complex64)

In [313]:
rasq - ras

<jax.Array float32(10, 10) ≈1.4 ±4.2 [≥-1.4e-06, ≤2.5e+01] zero:54 nonzero:46
  <Arrayviz rendering>
| Device: GPU 0>

## These Block Jacobis don't generalize wrt number of layers 

Need a coarse space from classical theory; so we build the coarse space; there's some 

In [314]:
phis = np.ones((L + 1, 2))
for i in range(L + 1): 
    phis[i, 1] = -1 + 2 * i / L
phis

array([[ 1.        , -1.        ],
       [ 1.        , -0.77777778],
       [ 1.        , -0.55555556],
       [ 1.        , -0.33333333],
       [ 1.        , -0.11111111],
       [ 1.        ,  0.11111111],
       [ 1.        ,  0.33333333],
       [ 1.        ,  0.55555556],
       [ 1.        ,  0.77777778],
       [ 1.        ,  1.        ]])

In [315]:
D1

array([[1. , 0. , 0. , 0. ],
       [0. , 1. , 0. , 0. ],
       [0. , 0. , 1. , 0. ],
       [0. , 0. , 0. , 0.5]])

In [316]:
Z = np.zeros((12, 6))
Z[0:4, 0:2] =  D1 @ R1 @ phis
Z[4:8, 2:4] =  D2 @ R2 @ phis
Z[8:12, 4:6] = D3 @ R3 @ phis
R0 = Z.T
R0

array([[ 1.        ,  1.        ,  1.        ,  0.5       ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-1.        , -0.77777778, -0.55555556, -0.16666667,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.5       ,
         1.        ,  1.        ,  0.5       ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -0.16666667,
        -0.11111111,  0.11111111,  0.16666667,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.5       ,  1.        ,
         1.        ,  1.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.16666667,  0.55555556,
         0.77777778,  1.        ]])

In [317]:
dgdu @ R0

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 6 is different from 10)

In [318]:
z = np.zeros((10, 3))
for i in range(ND): 
    ones = np.ones((10,))
    z_i = restrictions[i].T @ pou[i] @ restrictions[i] @ ones
    z[:, i] = z_i
z
R0 = z.T

In [319]:

rasq_wc = R0.T @ np.linalg.inv(R0 @ dgdu.T @ dgdu @ R0.T) @ R0 + rasq
# rasq_wc = rasq @ (R0.T @ np.linalg.inv(R0 @ dgdu.T @ dgdu @ R0.T) @ R0)

In [320]:
np.sort(np.linalg.eigvals((R0.T @ np.linalg.inv(R0 @ dgdu.T @ dgdu @ R0.T) @ R0) @  dgdu.T @ dgdu))

array([-1.86073390e-16, -1.11022302e-16, -3.22352237e-17,  1.41473080e-35,
        3.51791882e-17,  7.13545666e-17,  1.47632028e-16,  1.00000000e+00,
        1.00000000e+00,  1.00000000e+00])

In [321]:
np.sort(np.linalg.eigvals((rasq_wc) @  dgdu.T @ dgdu))

array([2.14034618e-02, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
       1.08675574e+00, 1.41769488e+00, 1.70457311e+00, 1.99950986e+00,
       7.43220224e+00, 4.53168082e+01])